# libraries

In [4]:
import os
import joblib
import json
import numpy as np
import flwr as fl
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import LabelEncoder
from flwr.common import parameters_to_ndarrays, ndarrays_to_parameters

# Config

In [5]:
MODEL_DIR = "models"
MODEL_PATH = os.path.join(MODEL_DIR, "network_sgd_classifier.joblib")
LE_PATH = os.path.join(MODEL_DIR, "label_encoder.joblib")
FEATURE_COUNT = 13
NUM_ROUNDS = 3
NUM_CLIENTS = 1
SERVER_ADDRESS = "0.0.0.0:8080"


# **Label encoder (server broadcasts classes_json to clients)**

In [6]:
if os.path.exists(LE_PATH):
    le = joblib.load(LE_PATH)
else:
    le = LabelEncoder()
    le.classes_ = np.array(["Benign", "Attack"])
    joblib.dump(le, LE_PATH)
    print(f"Saved placeholder label encoder to {LE_PATH}.")

# Create/load initial model
if os.path.exists(MODEL_PATH):
    model = joblib.load(MODEL_PATH)
    print(f"Loaded initial model from {MODEL_PATH}")
else:
    model = SGDClassifier(
        loss="log_loss", max_iter=1, tol=None, warm_start=True, random_state=42
    )
    dummy_X = np.zeros((1, FEATURE_COUNT))
    dummy_y = np.array([0])
    model.partial_fit(dummy_X, dummy_y, classes=np.arange(len(le.classes_)))
    joblib.dump(model, MODEL_PATH)
    print(f"Created and saved initial model to {MODEL_PATH}")
    
    
def get_model_parameters(m):
    return [m.coef_.astype(np.float32), m.intercept_.astype(np.float32)]


initial_parameters = ndarrays_to_parameters(get_model_parameters(model))



Loaded initial model from models\network_sgd_classifier.joblib


# Login & Update Aggration

In [ ]:

class LoggingFedAvg(fl.server.strategy.FedAvg):
    def __init__(
        self,
        *args,
        model_obj=None,
        model_dir=MODEL_DIR,
        initial_ndarrays=None,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.model_obj = model_obj
        self.model_dir = model_dir
        os.makedirs(self.model_dir, exist_ok=True)
        self._last_ndarrays = (
            initial_ndarrays
            if initial_ndarrays is not None
            else parameters_to_ndarrays(initial_parameters)
        )

    def _make_config(self, server_round):
        # IMPORTANT: do NOT put lists in the config. Serialize lists to JSON strings.
        return {
            "rnd": server_round,
            "classes_json": json.dumps(list(le.classes_)),  # string safe for protobuf
            "n_features": int(model.coef_.shape[1]),
        }

    def configure_fit(self, server_round, parameters, client_manager):
        config = self._make_config(server_round)
        fit_ins = fl.common.FitIns(parameters, config)
        clients = client_manager.sample(
            num_clients=self.min_fit_clients, min_num_clients=self.min_available_clients
        )
        return [(client, fit_ins) for client in clients]

    def configure_evaluate(self, server_round, parameters, client_manager):
        # Send same config for evaluate so clients can parse classes/n_features
        config = self._make_config(server_round)
        evaluate_ins = fl.common.EvaluateIns(parameters, config)
        clients = client_manager.sample(
            num_clients=self.min_evaluate_clients,
            min_num_clients=self.min_available_clients,
        )
        return [(client, evaluate_ins) for client in clients]

    def aggregate_fit(self, server_round, results, failures):
        n_clients = len(results)
        print(
            f"[Server] aggregate_fit called for round {server_round} with {n_clients} client updates (failures={len(failures)})"
        )

        if failures:
            print("[Server] Failures details:")
            for i, f in enumerate(failures):
                try:
                    print(f"  Failure #{i}: {repr(f)}")
                except Exception:
                    print(f"  Failure #{i}: (couldn't repr the failure)")

        aggregated = super().aggregate_fit(server_round, results, failures)

        if aggregated is None:
            print(
                "[Server] No aggregated parameters returned (None). Skipping persistence for this round."
            )
            return None

        # Parent may return (Parameters, something) or Parameters directly
        if isinstance(aggregated, tuple) and len(aggregated) >= 1:
            aggregated_params = aggregated[0]
        else:
            aggregated_params = aggregated

        try:
            new_ndarrays = parameters_to_ndarrays(aggregated_params)
        except Exception as e:
            print(f"[Server] ERROR converting aggregated params to ndarrays: {e}")
            return aggregated

        # Diagnostic diffs
        try:
            for i, (prev, new) in enumerate(zip(self._last_ndarrays, new_ndarrays)):
                prev_arr = np.asarray(prev, dtype=np.float64)
                new_arr = np.asarray(new, dtype=np.float64)
                diff = new_arr - prev_arr
                l2 = float(np.linalg.norm(diff))
                max_abs = float(np.max(np.abs(diff)))
                print(
                    f"[Server] param #{i}: L2-change={l2:.6e}, max-abs-change={max_abs:.6e}"
                )
        except Exception as e:
            print(f"[Server] Warning computing diffs: {e}")

        # Persist aggregated model
        try:
            if self.model_obj is not None:
                coef_arr, intercept_arr = new_ndarrays
                self.model_obj.coef_ = np.array(coef_arr, dtype=np.float64)
                self.model_obj.intercept_ = np.array(intercept_arr, dtype=np.float64)
                fname = os.path.join(
                    self.model_dir, f"network_sgd_classifier_round{server_round}.joblib"
                )
                joblib.dump(self.model_obj, fname)
                print(f"[Server] Saved aggregated model to: {fname}")
            else:
                fname = os.path.join(self.model_dir, f"params_round{server_round}.npz")
                np.savez(fname, *new_ndarrays)
                print(f"[Server] Saved aggregated ndarray params to: {fname}")
        except Exception as e:
            print(f"[Server] WARNING: failed to persist aggregated model: {e}")

        self._last_ndarrays = new_ndarrays
        return aggregated


strategy = LoggingFedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=NUM_CLIENTS,
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    initial_parameters=initial_parameters,
    model_obj=model,
    initial_ndarrays=parameters_to_ndarrays(initial_parameters),
)

if __name__ == "__main__":
    print(f"Starting Flower server at {SERVER_ADDRESS} for {NUM_ROUNDS} rounds ...")
    fl.server.start_server(
        server_address=SERVER_ADDRESS,
        config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
        strategy=strategy,
    )
    print("Server finished.")


	Instead, use the `flower-superlink` CLI command to start a SuperLink as shown below:

		$ flower-superlink --insecure

	To view usage and all available options, run:

		$ flower-superlink --help

	Using `start_server()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower server, config: num_rounds=3, no round_timeout
INFO :      Flower ECE: gRPC server running (3 rounds), SSL is disabled
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]


Starting Flower server at 0.0.0.0:8080 for 3 rounds ...
